# Data Exploration and Synthetic Dataset Generation

## Dataset Source
Synthetic dataset generated programmatically to simulate e-commerce complaints with:
- Complaint text (natural language)
- Tabular features (customer tenure, previous complaints)
- Product images (synthetic RGB images, 64x64x3)
- Risk label (low/medium/high)

## Dataset Shape
Total samples: 2000
Train: 1400 (70%)
Validation: 300 (15%)
Test: 300 (15%)

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from PIL import Image
import random

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Create data directory
os.makedirs('data', exist_ok=True)

n_samples = 2000

# Generate complaint texts
low_keywords = ['shipping delay', 'order status', 'when will arrive', 'tracking number', 'delivery date']
medium_keywords = ['wrong item', 'missing part', 'defective', 'poor quality', 'customer service slow']
high_keywords = ['broken', 'ignored', 'third time', 'refund', 'never replied', 'damaged', 'escalate', 'lawsuit']

complaints = []
risk_labels = []

for i in range(n_samples):
    # Determine risk based on weighted random
    rand = np.random.random()
    if rand < 0.5:
        risk = 'low'
        text = np.random.choice(low_keywords) + ' ' + ' '.join(np.random.choice(['please update', 'kindly advise'], 2))
    elif rand < 0.8:
        risk = 'medium'
        text = np.random.choice(medium_keywords) + ' ' + ' '.join(np.random.choice(['unacceptable', 'need resolution'], 2))
    else:
        risk = 'high'
        text = np.random.choice(high_keywords) + ' ' + ' '.join(np.random.choice(['no response', 'terrible experience'], 2))
    
    complaints.append(text)
    risk_labels.append(risk)

# Generate tabular features
customer_tenure = np.random.uniform(0, 60, n_samples)  # months
previous_complaints = np.random.poisson(lam=1.5, size=n_samples)

# Adjust risk based on tabular features (add realism)
for i in range(n_samples):
    if previous_complaints[i] > 3 and risk_labels[i] != 'high':
        risk_labels[i] = 'high'
    elif previous_complaints[i] > 1 and risk_labels[i] == 'low':
        risk_labels[i] = 'medium'

# Create dataframe
df = pd.DataFrame({
    'complaint_text': complaints,
    'customer_tenure': customer_tenure,
    'previous_complaints': previous_complaints,
    'risk_label': risk_labels
})

# Encode target
risk_mapping = {'low': 0, 'medium': 1, 'high': 2}
df['risk_encoded'] = df['risk_label'].map(risk_mapping)

# Generate synthetic images based on risk level
def generate_image(risk, size=(64, 64, 3)):
    img = np.random.randint(0, 255, size, dtype=np.uint8)
    # Add color pattern based on risk
    if risk == 'high':
        # Red tint
        img[:, :, 0] = np.clip(img[:, :, 0] + 100, 0, 255)
    elif risk == 'medium':
        # Yellow tint
        img[:, :, 1] = np.clip(img[:, :, 1] + 80, 0, 255)
    else:
        # Green tint
        img[:, :, 2] = np.clip(img[:, :, 2] + 60, 0, 255)
    return img

images = np.array([generate_image(df.iloc[i]['risk_label']) for i in range(n_samples)])

# Train/val/test split
X_text = df['complaint_text'].values
X_tabular = df[['customer_tenure', 'previous_complaints']].values
y = df['risk_encoded'].values

X_temp_text, X_test_text, X_temp_tab, X_test_tab, y_temp, y_test, img_temp, img_test = train_test_split(
    X_text, X_tabular, y, images, test_size=0.15, random_state=42, stratify=y
)

X_train_text, X_val_text, X_train_tab, X_val_tab, y_train, y_val, img_train, img_val = train_test_split(
    X_temp_text, X_temp_tab, y_temp, img_temp, test_size=0.15/0.85, random_state=42, stratify=y_temp
)

# Save data
np.save('data/X_train_images.npy', img_train)
np.save('data/X_val_images.npy', img_val)
np.save('data/X_test_images.npy', img_test)
np.save('data/y_train.npy', y_train)
np.save('data/y_val.npy', y_val)
np.save('data/y_test.npy', y_test)

train_df = pd.DataFrame({'text': X_train_text, 'tenure': X_train_tab[:,0], 'prev_complaints': X_train_tab[:,1], 'risk': y_train})
val_df = pd.DataFrame({'text': X_val_text, 'tenure': X_val_tab[:,0], 'prev_complaints': X_val_tab[:,1], 'risk': y_val})
test_df = pd.DataFrame({'text': X_test_text, 'tenure': X_test_tab[:,0], 'prev_complaints': X_test_tab[:,1], 'risk': y_test})

train_df.to_csv('data/train_df.csv', index=False)
val_df.to_csv('data/val_df.csv', index=False)
test_df.to_csv('data/test_df.csv', index=False)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print("Class distribution:")
print(df['risk_label'].value_counts())
print("\nMissing values:", df.isnull().sum().sum())

In [ ]:
# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(data=df, x='risk_label', ax=axes[0])
axes[0].set_title('Risk Class Distribution')
sns.histplot(data=df, x='customer_tenure', hue='risk_label', ax=axes[1])
axes[1].set_title('Tenure by Risk')
sns.histplot(data=df, x='previous_complaints', hue='risk_label', discrete=True, ax=axes[2])
axes[2].set_title('Previous Complaints by Risk')
plt.tight_layout()
plt.show()

# Show sample images
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for i, risk in enumerate(['low', 'medium', 'high']):
    idx = df[df['risk_label']==risk].index[0]
    axes[i].imshow(images[idx])
    axes[i].set_title(f'Risk: {risk}')
    axes[i].axis('off')
plt.show()